# AttCM-Alex Rigorous Paper Reproduction (Google Colab)
### **Scientific Reports (2025) 15:25408 | DOI: 10.1038/s41598-025-10537-6**
**“AI-driven smart agriculture using hybrid transformer-CNN for real time disease detection in sustainable farming”**

---
### **Scientific Audit & Architectural Corrections:**
1. **3 AttCM Modules (Figure 4 & Page 10)**: The paper replaces all three 3x3 convolutional layers (Conv3, Conv4, Conv5) with three AttCM modules (192 $\to$ 384, 384 $\to$ 256, 256 $\to$ 256).
2. **Spatial Shift-Aggregation (Equations 5–7)**: Direct implementation of spatial shifting across the 9 directions $(\Delta y, \Delta x) \in \{-1, 0, 1\}^2$ with sum aggregation, alongside the author repository's stacked 3x3 convolution option.
3. **Figure 4 Classifier Architecture**: Reconstructs the exact multi-stage dense layers: `9216 -> 4096 -> 2048 -> 1024 -> 512 -> num_classes` with `Dropout(0.5)`.
4. **Exact Label Smoothing (Equation 11)**: Implements $y_k^{LS} = y_k(1 - \alpha) + \frac{\alpha}{K}$ with $\alpha = 0.1$.
5. **No Test-Set Leakage**: Training runs strictly on the training set for 50 epochs. Checkpoint `last.pt` (Epoch 50) is evaluated **once** on the held-out test set after training completes.
6. **Dataset Provenance & Discrepancies**:
   - **Cucumber**: Reconstructed split: 664 usable images (320 Healthy, 344 Sick) vs 679 paper images (delta = 15 images).
   - **Banana**: Labeled `['Healthy', 'Sigatoka', 'Other_Banana_Disease']` with explicit `DATASET REPRODUCTION GAP` documentation (public BananaLSD lacks Bacterial wilt / Xanthomonas).
   - **Tomato**: Subsampled according to Table 4 category counts (3,822 images) with exactly 50 test images per category.
7. **In-Memory Caching**: Preloads resized tensors into RAM (`CachedImageDataset`) to bypass Colab disk I/O latency.
8. **Pre-Training Empirical Benchmark**: Runs a fast 1-epoch / 5-batch dry-run to measure actual GPU throughput before launching 50 epochs.

In [1]:
# 1. Environment & GPU Verification
!nvidia-smi

import os, sys, math, time, random, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CyclicLR
import torchvision.models as models
import torchvision.transforms as T
from torchvision.transforms import InterpolationMode
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"\nPyTorch Version: {torch.__version__}")
print(f"Active Device:   {device}")
if torch.cuda.is_available():
    print(f"GPU Name:        {torch.cuda.get_device_name(0)}")
    print(f"VRAM Available:  {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

Sun Oct  4 21:57:00 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Ingest Public Datasets via Kagglehub (No API Key Required)

In [2]:
!pip install -q kagglehub
import kagglehub

print("1. Downloading Cucumber Dataset (Mendeley Data / Kaggle mirror)...")
cuc_raw_path = kagglehub.dataset_download("sujaykapadnis/cucumber-disease-recognition-dataset")
print(f"   Downloaded to: {cuc_raw_path}")

print("\n2. Downloading Banana Leaf Spot Diseases Dataset (BananaLSD)...")
ban_raw_path = kagglehub.dataset_download("shifatearman/bananalsd")
print(f"   Downloaded to: {ban_raw_path}")

print("\n3. Downloading PlantVillage Dataset (Tomato classes)...")
tom_raw_path = kagglehub.dataset_download("abdallahalidev/plantvillage-dataset")
print(f"   Downloaded to: {tom_raw_path}")

1. Downloading Cucumber Dataset (Mendeley Data / Kaggle mirror)...


100%|██████████| 2.59G/2.59G [00:14<00:00, 193MB/s]


Extracting files...
   Downloaded to: /root/.cache/kagglehub/datasets/sujaykapadnis/cucumber-disease-recognition-dataset/versions/1

2. Downloading Banana Leaf Spot Diseases Dataset (BananaLSD)...


100%|██████████| 36.9M/36.9M [00:00<00:00, 132MB/s]

Extracting files...


   Downloaded to: /root/.cache/kagglehub/datasets/shifatearman/bananalsd/versions/1

3. Downloading PlantVillage Dataset (Tomato classes)...
Using Colab cache for faster access to the 'plantvillage-dataset' dataset.
   Downloaded to: /kaggle/input/plantvillage-dataset


## 3. Faithful Architectural Reconciliation & Inspection
Reconstructs:
1. **SpatialShiftAggregation**: Paper Equations (5), (6), and (7) spatial shift across 9 displacement vectors $(\Delta y, \Delta x) \in \{-1, 0, 1\}^2$ followed by sum aggregation.
2. **AttCM Module**: Combining 1x1 conv feature projections (64 $\to$ 128 $\to$ out_channels), convolution branch, scaled dot-product attention (Eq. 9), and learnable $\alpha, \beta$ scalars (Eq. 8).
3. **AttCMAlexThreeModules**: AlexNet backbone with three 3x3 convs replaced by 3 AttCM modules and Figure 4 classifier (`9216 -> 4096 -> 2048 -> 1024 -> 512 -> num_classes`).
4. **AttCMAlexAuthorVariant**: Author's single-AttCM script variant for comparison.

In [3]:
class SpatialShiftAggregation(nn.Module):
    """
    Direct implementation of Paper Equations (5), (6), and (7):
      1. Fully connected network generating 9 feature groups (k=3, k^2=9):
         \tilde{g}_{(p,q)} for p, q in {0, 1, 2}
      2. Spatial shift:
         g_{(p,q)i,j} = Shift(\tilde{g}_{(p,q)i,j}, p - \lfloor k/2 \rfloor, q - \lfloor k/2 \rfloor)
         where displacements (dy, dx) in {-1, 0, 1}^2
      3. Aggregation across the 9 shifted feature maps:
         F_convolution = \sum_{p,q} g_{(p,q)}
    """
    def __init__(self, in_channels, out_channels):
        super(SpatialShiftAggregation, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.fc = nn.Conv2d(in_channels, out_channels * 9, kernel_size=1)
        self.act = nn.ReLU(inplace=True)

    def forward(self, x):
        B, _, H, W = x.size()
        feats = self.act(self.fc(x)).view(B, 9, self.out_channels, H, W)

        shifts = [
            (-1, -1), (-1, 0), (-1, 1),
            (0, -1),  (0, 0),  (0, 1),
            (1, -1),  (1, 0),  (1, 1)
        ]

        out = torch.zeros(B, self.out_channels, H, W, device=x.device, dtype=x.dtype)
        for idx, (dy, dx) in enumerate(shifts):
            g = feats[:, idx]
            shifted_g = torch.zeros_like(g)

            src_y_start = max(0, -dy)
            src_y_end = H - max(0, dy)
            src_x_start = max(0, -dx)
            src_x_end = W - max(0, dx)

            dst_y_start = max(0, dy)
            dst_y_end = H - max(0, -dy)
            dst_x_start = max(0, dx)
            dst_x_end = W - max(0, -dx)

            shifted_g[:, :, dst_y_start:dst_y_end, dst_x_start:dst_x_end] = g[:, :, src_y_start:src_y_end, src_x_start:src_x_end]
            out = out + shifted_g

        return out


class AttCM(nn.Module):
    """
    Attention Convolution Mixed Module (Paper Section p. 10-12, Fig. 5, Eqs. 5-10)
    """
    def __init__(self, in_channels, out_channels, alpha_init=0.5, beta_init=0.5, conv_branch_mode="conv3x3"):
        super(AttCM, self).__init__()
        self.conv_branch_mode = conv_branch_mode

        # 1. Three sequential 1x1 convs (Paper p. 10 & Fig. 5)
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=1)
        self.conv2 = nn.Conv2d(64, 128, kernel_size=1)
        self.conv3 = nn.Conv2d(128, out_channels, kernel_size=1)

        # 2. Convolution branch: spatial shift-aggregation (Eqs. 5-7) or author stacked 3x3 convs
        if conv_branch_mode == "shift_aggregate":
            self.conv_branch = SpatialShiftAggregation(out_channels, out_channels)
        else:
            self.conv_branch = nn.Sequential(
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
            )

        # 3. Attention branch (Q, K, V projections and Scaled Dot-Product Attention Eq. 9)
        self.q_conv = nn.Conv2d(out_channels, out_channels, kernel_size=1)
        self.k_conv = nn.Conv2d(out_channels, out_channels, kernel_size=1)
        self.v_conv = nn.Conv2d(out_channels, out_channels, kernel_size=1)
        self.softmax = nn.Softmax(dim=-1)

        # Learnable fusion parameters (alpha, beta initialized to 0.5 from author code)
        self.alpha = nn.Parameter(torch.tensor(float(alpha_init)))
        self.beta = nn.Parameter(torch.tensor(float(beta_init)))

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))

        conv_out = self.conv_branch(x)

        B, C, H, W = x.size()
        N = H * W
        q = self.q_conv(x).view(B, C, N)
        k = self.k_conv(x).view(B, C, N).transpose(-2, -1)
        v = self.v_conv(x).view(B, C, N)

        scale = 1.0 / math.sqrt(C)
        attn_weights = self.softmax(torch.bmm(k, q) * scale)
        attn_out = torch.bmm(v, attn_weights).view(B, C, H, W)

        # Eq. 8: F_out = alpha * F_attention + beta * F_convolution
        return self.alpha * attn_out + self.beta * conv_out


class AttCMAlexThreeModules(nn.Module):
    """
    Paper Architecture (Page 10 & Fig. 4):
    - Conv1 (64) + MaxPool
    - Conv2 (192) + MaxPool
    - Three AttCM modules replacing Conv3, Conv4, and Conv5:
      * AttCM 1: 192 -> 384
      * AttCM 2: 384 -> 256
      * AttCM 3: 256 -> 256
    - MaxPool(3, s=2)
    - Multi-stage Classifier (Figure 4: 9216 -> 4096 -> 2048 -> 1024 -> 512 -> num_classes)
    """
    def __init__(self, num_classes=2, pretrained=True, conv_branch_mode="conv3x3"):
        super(AttCMAlexThreeModules, self).__init__()
        # Pretrained AlexNet weights adopted from author repository implementation
        weights = models.AlexNet_Weights.DEFAULT if pretrained else None
        alexnet = models.alexnet(weights=weights)
        features = list(alexnet.features.children())

        # Layer 1: Conv1 (3 -> 64, 11x11, s=4) + ReLU
        self.layer1 = nn.Sequential(*features[0:2])
        self.pool1 = features[2]

        # Layer 2: Conv2 (64 -> 192, 5x5) + ReLU
        self.layer2 = nn.Sequential(*features[3:5])
        self.pool2 = features[5]

        # Three AttCM modules replacing Conv3, Conv4, and Conv5
        self.attcm1 = AttCM(in_channels=192, out_channels=384, conv_branch_mode=conv_branch_mode)
        self.attcm2 = AttCM(in_channels=384, out_channels=256, conv_branch_mode=conv_branch_mode)
        self.attcm3 = AttCM(in_channels=256, out_channels=256, conv_branch_mode=conv_branch_mode)

        self.post_pool = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2)
        )

        flattened_dim = 256 * 6 * 6  # 9216

        # Multi-stage Classifier (Strictly follows Figure 4)
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(flattened_dim, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(4096, 2048),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(2048, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.pool1(x)
        x = self.layer2(x)
        x = self.pool2(x)
        x = self.attcm1(x)
        x = self.attcm2(x)
        x = self.attcm3(x)
        x = self.post_pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)


class AttCMAlexAuthorVariant(nn.Module):
    """
    Direct replication of author's uploaded script (AttCMAlex.py) for comparison:
    Contains only 1 AttCM and a truncated 3-layer classifier (2048 -> 1024 -> num_classes).
    """
    def __init__(self, num_classes=2, pretrained=True):
        super(AttCMAlexAuthorVariant, self).__init__()
        weights = models.AlexNet_Weights.DEFAULT if pretrained else None
        alexnet = models.alexnet(weights=weights)
        features = list(alexnet.features.children())
        self.layer1 = nn.Sequential(*features[0:2])
        self.pool1 = features[2]
        self.layer2 = nn.Sequential(*features[3:5])
        self.pool2 = features[5]
        self.attcm = AttCM(in_channels=192, out_channels=256, conv_branch_mode="conv3x3")
        self.post_attcm = nn.Sequential(
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Dropout(p=0.5),
            nn.Linear(256 * 6 * 6, 2048),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.5),
            nn.Linear(2048, 1024),
            nn.ReLU(inplace=True),
            nn.Linear(1024, num_classes)
        )

    def forward(self, x):
        x = self.layer1(x)
        x = self.pool1(x)
        x = self.layer2(x)
        x = self.pool2(x)
        x = self.attcm(x)
        x = self.post_attcm(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

# Architectural Audit & Diagnostic Report (No artificial assertion gates)
m_paper_conv = AttCMAlexThreeModules(num_classes=10, pretrained=False, conv_branch_mode="conv3x3")
m_paper_shift = AttCMAlexThreeModules(num_classes=10, pretrained=False, conv_branch_mode="shift_aggregate")
m_author = AttCMAlexAuthorVariant(num_classes=10, pretrained=False)
m_alex = models.alexnet()

# Trace tensor dimensions with dummy input
dummy = torch.randn(1, 3, 224, 224)
print("=" * 80)
print("           ARCHITECTURAL AUDIT & TENSOR DIMENSION TRACE")
print("=" * 80)
x = dummy
x = m_paper_conv.layer1(x); print(f"After Layer 1 (Conv1 11x11, s=4 + ReLU): {list(x.shape)}")
x = m_paper_conv.pool1(x);  print(f"After Pool 1 (MaxPool 3x3, s=2):          {list(x.shape)}")
x = m_paper_conv.layer2(x); print(f"After Layer 2 (Conv2 5x5 + ReLU):        {list(x.shape)}")
x = m_paper_conv.pool2(x);  print(f"After Pool 2 (MaxPool 3x3, s=2):          {list(x.shape)}")
x = m_paper_conv.attcm1(x); print(f"After AttCM 1 (192 -> 384):               {list(x.shape)}")
x = m_paper_conv.attcm2(x); print(f"After AttCM 2 (384 -> 256):               {list(x.shape)}")
x = m_paper_conv.attcm3(x); print(f"After AttCM 3 (256 -> 256):               {list(x.shape)}")
x = m_paper_conv.post_pool(x); print(f"After Post-AttCM Pool (MaxPool 3x3, s=2): {list(x.shape)}")
x = torch.flatten(x, 1);    print(f"Flattened Tensor:                        {list(x.shape)} (dim = 9216)")
print("=" * 80)
print("                      PARAMETER BREAKDOWN COMPARISON")
print("=" * 80)
print(f"1. AlexNet Baseline (Torchvision):                  {sum(p.numel() for p in m_alex.parameters())/1e6:6.2f} M params  (Paper Table 6: 54.16 M)")
print(f"2. Paper Faithful (3 AttCM + Stacked Conv3x3):      {sum(p.numel() for p in m_paper_conv.parameters())/1e6:6.2f} M params  (Paper Table 6: 66.24 M)")
print(f"3. Paper Faithful (3 AttCM + Shift-Agg Conv):       {sum(p.numel() for p in m_paper_shift.parameters())/1e6:6.2f} M params  (Eqs. 5-7 Shift Mechanism)")
print(f"4. Author Script Variant (1 AttCM Module):          {sum(p.numel() for p in m_author.parameters())/1e6:6.2f} M params  (Author AttCMAlex.py truncation)")
print("=" * 80)
print("Verified: Model implements 3 AttCM modules and Figure 4 classifier.")

<>:7: SyntaxWarning: invalid escape sequence '\l'
<>:7: SyntaxWarning: invalid escape sequence '\l'
/tmp/ipykernel_640/1370556084.py:7: SyntaxWarning: invalid escape sequence '\l'
  g_{(p,q)i,j} = Shift(\tilde{g}_{(p,q)i,j}, p - \lfloor k/2 \rfloor, q - \lfloor k/2 \rfloor)


           ARCHITECTURAL AUDIT & TENSOR DIMENSION TRACE
After Layer 1 (Conv1 11x11, s=4 + ReLU): [1, 64, 55, 55]
After Pool 1 (MaxPool 3x3, s=2):          [1, 64, 27, 27]
After Layer 2 (Conv2 5x5 + ReLU):        [1, 192, 27, 27]
After Pool 2 (MaxPool 3x3, s=2):          [1, 192, 13, 13]
After AttCM 1 (192 -> 384):               [1, 384, 13, 13]
After AttCM 2 (384 -> 256):               [1, 256, 13, 13]
After AttCM 3 (256 -> 256):               [1, 256, 13, 13]
After Post-AttCM Pool (MaxPool 3x3, s=2): [1, 256, 6, 6]
Flattened Tensor:                        [1, 9216] (dim = 9216)
                      PARAMETER BREAKDOWN COMPARISON
1. AlexNet Baseline (Torchvision):                   61.10 M params  (Paper Table 6: 54.16 M)
2. Paper Faithful (3 AttCM + Stacked Conv3x3):       55.15 M params  (Paper Table 6: 66.24 M)
3. Paper Faithful (3 AttCM + Shift-Agg Conv):        52.65 M params  (Eqs. 5-7 Shift Mechanism)
4. Author Script Variant (1 AttCM Module):           22.75 M params  (Author 

## 4. In-Memory Caching Dataset (Zero Virtual Disk Latency)
Caches resized $(3 \times 224 \times 224)$ FloatTensors in RAM to eliminate Colab filesystem read overhead during training.

In [4]:
transform_base = T.Compose([
    T.Resize((224, 224), interpolation=InterpolationMode.BILINEAR),
    T.ToTensor(),
])

class CachedImageDataset(Dataset):
    """
    Preloads all resized tensors into RAM.
    Eliminates Colab virtual disk I/O bottlenecks.
    """
    def __init__(self, samples, transform=transform_base, desc="Dataset"):
        self.samples = samples
        self.data = []
        print(f"Caching {len(samples)} images for {desc} into RAM...")
        for path, label in samples:
            try:
                with Image.open(path) as img:
                    img = img.convert('RGB')
                    tensor = transform(img)
                self.data.append((tensor, label))
            except Exception as e:
                pass
        print(f"  -> Successfully cached {len(self.data)} images into memory!")

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx]

## 5. Dataset Preparation & Provenance Audit
Audits splits and explicitly documents all reproduction gaps.

In [5]:
def find_files(folder):
    exts = ('.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG')
    files = []
    for root, _, filenames in os.walk(folder):
        for f in filenames:
            if f.endswith(exts):
                files.append(os.path.join(root, f))
    return sorted(files)

# 1. CUCUMBER DATASET
# Paper Stated (Table 2): 679 images (Healthy: 335, Sick: 344) with 80% train / 20% test.
# Available Source: 664 usable images (Fresh Leaf + Cucumber: 320, Sick: 344). Delta = 15 images.
def prepare_cucumber(base_dir, seed=42):
    random.seed(seed)
    bases = glob.glob(f"{base_dir}/**/Cucumber Disease Recognition Dataset/Original Image/Original Image", recursive=True)
    if not bases:
        bases = glob.glob(f"{base_dir}/**/Original Image/Original Image", recursive=True)
    b = bases[0] if bases else base_dir

    fresh = find_files(os.path.join(b, 'Fresh Leaf')) + find_files(os.path.join(b, 'Fresh Cucumber'))
    healthy = fresh[:335] if len(fresh) >= 335 else fresh

    sick_all = []
    for d in ['Anthracnose', 'Bacterial Wilt', 'Downy Mildew', 'Gummy Stem Blight', 'Belly Rot', 'Pythium Fruit Rot']:
        sick_all.extend(find_files(os.path.join(b, d)))
    sick = sick_all[:344] if len(sick_all) >= 344 else sick_all

    train_s, test_s = [], []
    for label_idx, files in enumerate([healthy, sick]):
        shuff = files.copy()
        random.shuffle(shuff)
        n_train = int(len(shuff) * 0.80)
        train_s.extend([(f, label_idx) for f in shuff[:n_train]])
        test_s.extend([(f, label_idx) for f in shuff[n_train:]])

    print("=" * 75)
    print("[CUCUMBER AUDIT] Reconstructed Dataset Split")
    print(f"  Paper Stated:     679 images (Healthy: 335, Sick: 344)")
    print(f"  Available Source: {len(healthy)+len(sick)} usable images (Healthy: {len(healthy)}, Sick: {len(sick)})")
    print(f"  Discrepancy:      {679 - (len(healthy)+len(sick))} missing healthy images in public source")
    print(f"  Split:            Train: {len(train_s)} | Held-Out Test: {len(test_s)}")
    print("=" * 75)
    return train_s, test_s, ['Healthy', 'Sick']

# 2. BANANA DATASET
# Paper Stated (Table 3): 1,284 images (Healthy: 150, Sigatoka: 320, Bacterial wilt/Xanthomonas: 814) with 20% test per class.
# Available Source (BananaLSD OriginalSet): 937 images (healthy: 129, sigatoka: 473, cordana: 162, pestalotiopsis: 173).
def prepare_banana(base_dir, seed=42):
    random.seed(seed)
    bases = glob.glob(f"{base_dir}/**/BananaLSD/OriginalSet", recursive=True)
    b = bases[0] if bases else base_dir

    h_files = find_files(os.path.join(b, 'healthy'))
    s_files = find_files(os.path.join(b, 'sigatoka'))
    # Cordana and Pestalotiopsis combined as Other_Banana_Disease
    c_files = find_files(os.path.join(b, 'cordana')) + find_files(os.path.join(b, 'pestalotiopsis'))

    classes = ['Healthy', 'Sigatoka', 'Other_Banana_Disease']
    train_s, test_s = [], []
    for label_idx, (name, files) in enumerate(zip(classes, [h_files, s_files, c_files])):
        shuff = files.copy()
        random.shuffle(shuff)
        n_test = max(1, int(len(shuff) * 0.20))
        train_s.extend([(f, label_idx) for f in shuff[:-n_test]])
        test_s.extend([(f, label_idx) for f in shuff[-n_test:]])

    print("=" * 75)
    print("[BANANA AUDIT] DATASET REPRODUCTION GAP")
    print(f"  Paper Stated:     1,284 images (Healthy: 150, Sigatoka: 320, Bacterial Wilt/Xanthomonas: 814)")
    print(f"  Available Source: {len(h_files)+len(s_files)+len(c_files)} images (Healthy: {len(h_files)}, Sigatoka: {len(s_files)}, Other: {len(c_files)})")
    print("  GAP NOTE:         Public BananaLSD does NOT contain Bacterial Wilt (Xanthomonas).")
    print("                    Class 3 is mapped to 'Other_Banana_Disease' (Cordana + Pestalotiopsis).")
    print(f"  Split:            Train: {len(train_s)} | Held-Out Test: {len(test_s)}")
    print("=" * 75)
    return train_s, test_s, classes

# 3. TOMATO DATASET
# Paper Stated: 4,021 total in text; Table 4 sums to 3,822 images across 10 classes. 50 test images per category.
# Available Source (PlantVillage): ~18,160 total tomato images across 10 classes.
def prepare_tomato(base_dir, seed=42):
    random.seed(seed)
    bases = glob.glob(f"{base_dir}/**/plantvillage dataset/color", recursive=True)
    if not bases:
        bases = glob.glob(f"{base_dir}/**/color", recursive=True)
    b = bases[0]

    class_map = [
        ('Yellow_leaf_curl_virus', 'Tomato___Tomato_Yellow_Leaf_Curl_Virus'),
        ('Bacterial_spot', 'Tomato___Bacterial_spot'),
        ('Mosaic_virus', 'Tomato___Tomato_mosaic_virus'),
        ('Late_blight', 'Tomato___Late_blight'),
        ('Septoria_leaf_spot', 'Tomato___Septoria_leaf_spot'),
        ('Spider_mites', 'Tomato___Spider_mites Two-spotted_spider_mite'),
        ('Early_blight', 'Tomato___Early_blight'),
        ('Target_spot', 'Tomato___Target_Spot'),
        ('Leaf_mold', 'Tomato___Leaf_Mold'),
        ('Healthy', 'Tomato___healthy'),
    ]

    # Subsample to paper Table 4 category sizes (sum = 3,822)
    paper_subsamples = [389, 287, 384, 417, 369, 406, 361, 401, 395, 413]
    classes = [c[0] for c in class_map]
    train_s, test_s = [], []

    for label_idx, ((name, folder_name), target_n) in enumerate(zip(class_map, paper_subsamples)):
        p = os.path.join(b, folder_name)
        files = find_files(p)
        random.shuffle(files)
        sampled = files[:target_n] if len(files) >= target_n else files
        # Paper states exactly 50 test images per category
        n_test = min(50, len(sampled) // 2)
        train_s.extend([(f, label_idx) for f in sampled[:-n_test]])
        test_s.extend([(f, label_idx) for f in sampled[-n_test:]])

    print("=" * 75)
    print("[TOMATO AUDIT] Reconstructed Dataset Split")
    print(f"  Paper Stated:     4,021 text / 3,822 Table 4 sum")
    print(f"  Sampled Subset:   {len(train_s)+len(test_s)} images across 10 categories")
    print(f"  Split:            Train: {len(train_s)} | Held-Out Test: {len(test_s)} (exactly 50 per category)")
    print("=" * 75)
    return train_s, test_s, classes

## 6. Pre-Training Empirical Throughput Benchmark
Measures actual forward/backward pass throughput on the active T4 GPU instance before committing to 50 epochs.

In [6]:
def benchmark_gpu_throughput(model, sample_batch_size=32, num_batches=10):
    model = model.to(device)
    model.train()
    optimizer = AdamW(model.parameters(), lr=1e-3)
    criterion = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    # Synthetic batch of 224x224 images
    x = torch.randn(sample_batch_size, 3, 224, 224, device=device)
    y = torch.zeros(sample_batch_size, dtype=torch.long, device=device)

    # Warmup
    for _ in range(3):
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                out = model(x)
                loss = criterion(out, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    # Timed run
    t0 = time.time()
    for _ in range(num_batches):
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                out = model(x)
                loss = criterion(out, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.time() - t0

    sec_per_batch = elapsed / num_batches
    img_per_sec = (sample_batch_size * num_batches) / elapsed
    print("=" * 75)
    print("                 EMPIRICAL T4 THROUGHPUT BENCHMARK")
    print("=" * 75)
    print(f"  Batch Size:            {sample_batch_size}")
    print(f"  Step Time (fwd + bwd): {sec_per_batch*1000:.1f} ms / batch")
    print(f"  Throughput:            {img_per_sec:.1f} images / second")
    print(f"  Estimated Epoch Times with RAM Caching:")
    print(f"    - Cucumber (531 train images): ~{531 / img_per_sec:.1f} seconds / epoch")
    print(f"    - Banana   (750 train images): ~{750 / img_per_sec:.1f} seconds / epoch")
    print(f"    - Tomato  (3322 train images): ~{3322 / img_per_sec:.1f} seconds / epoch")
    print("=" * 75)

test_model = AttCMAlexThreeModules(num_classes=2, pretrained=False, conv_branch_mode="conv3x3")
benchmark_gpu_throughput(test_model)

                 EMPIRICAL T4 THROUGHPUT BENCHMARK
  Batch Size:            32
  Step Time (fwd + bwd): 56.4 ms / batch
  Throughput:            567.4 images / second
  Estimated Epoch Times with RAM Caching:
    - Cucumber (531 train images): ~0.9 seconds / epoch
    - Banana   (750 train images): ~1.3 seconds / epoch
    - Tomato  (3322 train images): ~5.9 seconds / epoch


## 7. Training Pipeline (AdamW, CyclicLR, Equation 11 Loss, No Test-Set Leakage)

In [7]:
# Exact Label Smoothing Loss (Paper Equation 11: y^{LS}_k = y_k * (1 - alpha) + alpha / K)
class LabelSmoothingCrossEntropy(nn.Module):
    """
    Paper Equation (11):
      y^{LS}_k = y_k * (1 - alpha) + alpha / K
    Target class: (1 - alpha + alpha / K), Non-target classes: alpha / K
    """
    def __init__(self, alpha=0.1):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.alpha = alpha

    def forward(self, inputs, targets):
        targets = targets.view(-1)
        K = inputs.size(1)
        with torch.no_grad():
            true_dist = torch.full_like(inputs, self.alpha / K)
            true_dist.scatter_(1, targets.unsqueeze(1), 1.0 - self.alpha + self.alpha / K)
        return -(true_dist * F.log_softmax(inputs, dim=1)).sum(dim=1).mean()


def train_pipeline(crop_name, train_samples, test_samples, classes, epochs=50, batch_size=32,
                   pretrained=True, conv_branch_mode="conv3x3", step_mode="batch"):
    set_seed(42)
    num_classes = len(classes)

    # Preload samples into RAM
    train_ds = CachedImageDataset(train_samples, desc=f"{crop_name.capitalize()} Train")
    test_ds = CachedImageDataset(test_samples, desc=f"{crop_name.capitalize()} Test")
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, pin_memory=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, pin_memory=True)

    # 3 AttCM Modules Model
    model = AttCMAlexThreeModules(num_classes=num_classes, pretrained=pretrained, conv_branch_mode=conv_branch_mode).to(device)
    criterion = LabelSmoothingCrossEntropy(alpha=0.1)

    # Optimizer & Scheduler (Author-repository hyperparameters, Table 5 specs)
    optimizer = AdamW(model.parameters(), lr=1e-3, weight_decay=0.01)
    total_steps = len(train_loader) * epochs if step_mode == "batch" else epochs
    step_size_up = max(10, min(2000, total_steps // 2))
    scheduler = CyclicLR(optimizer, base_lr=1e-6, max_lr=1e-3, step_size_up=step_size_up, mode='triangular2', cycle_momentum=False)
    scaler = torch.amp.GradScaler('cuda') if device.type == 'cuda' else None

    os.makedirs(f'checkpoints/{crop_name}', exist_ok=True)
    os.makedirs(f'results/{crop_name}', exist_ok=True)

    history = []
    print(f"\n{'='*75}\n>>> Training {crop_name.upper()} (3 AttCM Modules | {epochs} Epochs | Pretrained: {pretrained})\n{'='*75}")

    for epoch in range(epochs):
        t0 = time.time()
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(device, non_blocking=True), targets.to(device, non_blocking=True)
            optimizer.zero_grad()

            if scaler:
                with torch.amp.autocast('cuda'):
                    outputs = model(inputs)
                    loss = criterion(outputs, targets)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                outputs = model(inputs)
                loss = criterion(outputs, targets)
                loss.backward()
                optimizer.step()

            if step_mode == "batch":
                scheduler.step()

            running_loss += loss.item() * inputs.size(0)
            correct += outputs.argmax(1).eq(targets).sum().item()
            total += targets.size(0)

        if step_mode == "epoch":
            scheduler.step()

        train_loss = running_loss / total
        train_acc = correct / total
        duration = time.time() - t0

        if (epoch + 1) % 5 == 0 or epoch == 0 or (epoch + 1) == epochs:
            print(f"Epoch [{epoch+1:2d}/{epochs:2d}] Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f} | {duration:.2f}s")

        history.append({
            'epoch': epoch + 1,
            'train_loss': train_loss, 'train_acc': train_acc,
            'duration_s': duration
        })

    # Save final Epoch 50 model checkpoint
    torch.save(model.state_dict(), f'checkpoints/{crop_name}/last.pt')
    pd.DataFrame(history).to_csv(f'results/{crop_name}/training_history.csv', index=False)

    # UNBIASED POST-TRAINING TEST EVALUATION (Evaluated ONCE after 50 epochs; NO test-set leakage)
    print(f"\n{'='*60}\n[Held-Out Test Evaluation] Evaluating Epoch 50 checkpoint for {crop_name.upper()}...\n{'='*60}")
    model.eval()
    test_loss, test_corr, test_tot = 0.0, 0, 0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for inputs, targets in test_loader:
            inputs, targets = inputs.to(device, non_blocking=True), targets.to(device, non_blocking=True)
            if scaler:
                with torch.amp.autocast('cuda'):
                    outputs = model(inputs)
                    loss = criterion(outputs, targets)
            else:
                outputs = model(inputs)
                loss = criterion(outputs, targets)
            test_loss += loss.item() * inputs.size(0)
            preds = outputs.argmax(1)
            test_corr += preds.eq(targets).sum().item()
            test_tot += targets.size(0)
            all_preds.extend(preds.cpu().tolist())
            all_targets.extend(targets.cpu().tolist())

    test_acc = test_corr / test_tot
    test_loss /= test_tot
    print(f">>> {crop_name.upper()} Final Held-Out Test Accuracy: {test_acc * 100:.2f}% | Test Loss: {test_loss:.4f}")
    return model, test_loader, test_acc

## 8. Execute Full 50-Epoch Training for Cucumber, Banana, and Tomato

In [8]:
summary = []

# 1. Cucumber (Target: 95.3%)
c_train, c_test, c_classes = prepare_cucumber(cuc_raw_path)
c_model, c_loader, c_acc = train_pipeline('cucumber', c_train, c_test, c_classes, epochs=50)
summary.append({'Crop': 'Cucumber', 'Classes': len(c_classes), 'Images': len(c_train) + len(c_test), 'Paper Target': '95.3%', 'Reproduction Test Acc': f"{c_acc*100:.2f}%"})

# 2. Banana (Target: 97.6%)
b_train, b_test, b_classes = prepare_banana(ban_raw_path)
b_model, b_loader, b_acc = train_pipeline('banana', b_train, b_test, b_classes, epochs=50)
summary.append({'Crop': 'Banana (Gap Note)', 'Classes': len(b_classes), 'Images': len(b_train) + len(b_test), 'Paper Target': '97.6%', 'Reproduction Test Acc': f"{b_acc*100:.2f}%"})

# 3. Tomato (Target: 97.9%)
t_train, t_test, t_classes = prepare_tomato(tom_raw_path)
t_model, t_loader, t_acc = train_pipeline('tomato', t_train, t_test, t_classes, epochs=50)
summary.append({'Crop': 'Tomato', 'Classes': len(t_classes), 'Images': len(t_train) + len(t_test), 'Paper Target': '97.9%', 'Reproduction Test Acc': f"{t_acc*100:.2f}%"})

# Summary Table
df_summary = pd.DataFrame(summary)
df_summary.to_csv('results/reproduction_summary.csv', index=False)
print("\n" + "=" * 75)
print("                   PAPER REPRODUCTION SUMMARY TABLE")
print("=" * 75)
display(df_summary)

[CUCUMBER AUDIT] Reconstructed Dataset Split
  Paper Stated:     679 images (Healthy: 335, Sick: 344)
  Available Source: 664 usable images (Healthy: 320, Sick: 344)
  Discrepancy:      15 missing healthy images in public source
  Split:            Train: 531 | Held-Out Test: 133
Caching 531 images for Cucumber Train into RAM...
  -> Successfully cached 531 images into memory!
Caching 133 images for Cucumber Test into RAM...
  -> Successfully cached 133 images into memory!
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 170MB/s]



>>> Training CUCUMBER (3 AttCM Modules | 50 Epochs | Pretrained: True)
Epoch [ 1/50] Train Loss: 0.6931 | Train Acc: 0.5160 | LR: 0.000041 | 1.49s
Epoch [ 5/50] Train Loss: 0.6932 | Train Acc: 0.5141 | LR: 0.000201 | 1.01s
Epoch [10/50] Train Loss: 0.6929 | Train Acc: 0.5179 | LR: 0.000401 | 1.00s
Epoch [15/50] Train Loss: 0.6928 | Train Acc: 0.5179 | LR: 0.000600 | 1.00s
Epoch [20/50] Train Loss: 0.4636 | Train Acc: 0.8493 | LR: 0.000800 | 1.01s
Epoch [25/50] Train Loss: 0.4336 | Train Acc: 0.8757 | LR: 0.001000 | 1.03s
Epoch [30/50] Train Loss: 0.4307 | Train Acc: 0.9002 | LR: 0.000800 | 1.02s
Epoch [35/50] Train Loss: 0.3234 | Train Acc: 0.9379 | LR: 0.000600 | 1.02s
Epoch [40/50] Train Loss: 0.2697 | Train Acc: 0.9699 | LR: 0.000401 | 1.01s
Epoch [45/50] Train Loss: 0.2251 | Train Acc: 0.9868 | LR: 0.000201 | 1.02s
Epoch [50/50] Train Loss: 0.2136 | Train Acc: 0.9944 | LR: 0.000001 | 1.02s

[Held-Out Test Evaluation] Evaluating Epoch 50 checkpoint for CUCUMBER...
>>> CUCUMBER Fina

/usr/local/lib/python3.13/dist-packages/PIL/TiffImagePlugin.py:950: UserWarning: Truncated File Read
  warnings.warn(str(msg))
/usr/local/lib/python3.13/dist-packages/PIL/JpegImagePlugin.py:887: UserWarning: Image appears to be a malformed MPO file, it will be interpreted as a base JPEG file
  warnings.warn(


  -> Successfully cached 751 images into memory!
Caching 186 images for Banana Test into RAM...
  -> Successfully cached 186 images into memory!

>>> Training BANANA (3 AttCM Modules | 50 Epochs | Pretrained: True)
Epoch [ 1/50] Train Loss: 1.1002 | Train Acc: 0.2783 | LR: 0.000041 | 1.63s
Epoch [ 5/50] Train Loss: 0.8024 | Train Acc: 0.5233 | LR: 0.000201 | 1.48s
Epoch [10/50] Train Loss: 0.6646 | Train Acc: 0.7909 | LR: 0.000401 | 1.44s
Epoch [15/50] Train Loss: 0.4975 | Train Acc: 0.9268 | LR: 0.000600 | 1.79s
Epoch [20/50] Train Loss: 0.4925 | Train Acc: 0.9401 | LR: 0.000800 | 1.44s
Epoch [25/50] Train Loss: 0.6772 | Train Acc: 0.8256 | LR: 0.001000 | 1.46s
Epoch [30/50] Train Loss: 0.4092 | Train Acc: 0.9574 | LR: 0.000800 | 1.44s
Epoch [35/50] Train Loss: 0.3424 | Train Acc: 0.9867 | LR: 0.000600 | 1.44s
Epoch [40/50] Train Loss: 0.3259 | Train Acc: 0.9907 | LR: 0.000401 | 1.47s
Epoch [45/50] Train Loss: 0.3165 | Train Acc: 0.9920 | LR: 0.000201 | 1.46s
Epoch [50/50] Train Loss:

,Crop,Classes,Images,Paper Target,Reproduction Test Acc
0,Cucumber,2,664,95.3%,90.23%
1,Banana (Gap Note),3,937,97.6%,97.85%
2,Tomato,10,3811,97.9%,84.40%


## 9. Robustness Evaluation Suite (Paper Section 3, Tables 7 & 8)
Evaluates the final 50-epoch checkpoints under:
- Salt-and-Pepper Noise (10%, 20%, 30%)
- Brightness Variations ($\pm 10\%, \pm 20\%, \pm 30\%$)
- Grayscale Luminance Conversion

In [9]:
def eval_robustness_suite(model, loader, crop_name):
    model.eval()
    print(f"\n{'='*75}\n[Robustness Suite: {crop_name.upper()}]\n{'='*75}")

    # 1. Salt-and-Pepper Noise (Paper Table 7)
    for ratio in [0.10, 0.20, 0.30]:
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, targets in loader:
                noisy = torch.stack([
                    (lambda t: (t.clone().masked_fill_(
                        torch.rand_like(t[0:1]) < (ratio/2), 0.0
                    ).masked_fill_(
                        torch.rand_like(t[0:1]) < (ratio/2), 1.0
                    )))(img) for img in inputs
                ]).to(device)
                preds = model(noisy).argmax(1).cpu()
                correct += preds.eq(targets).sum().item()
                total += targets.size(0)
        print(f"  Salt-and-Pepper Noise ({int(ratio*100)}%): Accuracy = {correct/total*100:.2f}%")

    # 2. Brightness Variations (Paper Table 8)
    for delta in [-0.30, -0.20, -0.10, 0.0, 0.10, 0.20, 0.30]:
        correct, total = 0, 0
        with torch.no_grad():
            for inputs, targets in loader:
                bright = torch.clamp(inputs * (1.0 + delta), 0.0, 1.0).to(device)
                preds = model(bright).argmax(1).cpu()
                correct += preds.eq(targets).sum().item()
                total += targets.size(0)
        sign = '+' if delta > 0 else ''
        label = f"{sign}{int(delta*100)}%" if delta != 0 else "0%"
        print(f"  Brightness Variation ({label:>4s}): Accuracy = {correct/total*100:.2f}%")

    # 3. Grayscale Luminance (Paper Table 8)
    correct, total = 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            gray = (0.2989*inputs[:, 0:1] + 0.5870*inputs[:, 1:2] + 0.1140*inputs[:, 2:3]).repeat(1, 3, 1, 1).to(device)
            preds = model(gray).argmax(1).cpu()
            correct += preds.eq(targets).sum().item()
            total += targets.size(0)
    print(f"  Grayscale Luminance:           Accuracy = {correct/total*100:.2f}%")

eval_robustness_suite(c_model, c_loader, 'cucumber')
eval_robustness_suite(b_model, b_loader, 'banana')
eval_robustness_suite(t_model, t_loader, 'tomato')


[Robustness Suite: CUCUMBER]
  Salt-and-Pepper Noise (10%): Accuracy = 68.42%
  Salt-and-Pepper Noise (20%): Accuracy = 48.12%
  Salt-and-Pepper Noise (30%): Accuracy = 48.12%
  Brightness Variation (-30%): Accuracy = 86.47%
  Brightness Variation (-20%): Accuracy = 89.47%
  Brightness Variation (-10%): Accuracy = 91.73%
  Brightness Variation (  0%): Accuracy = 90.23%
  Brightness Variation (+10%): Accuracy = 88.72%
  Brightness Variation (+20%): Accuracy = 88.72%
  Brightness Variation (+30%): Accuracy = 87.97%
  Grayscale Luminance:           Accuracy = 71.43%

[Robustness Suite: BANANA]
  Salt-and-Pepper Noise (10%): Accuracy = 43.55%
  Salt-and-Pepper Noise (20%): Accuracy = 17.20%
  Salt-and-Pepper Noise (30%): Accuracy = 13.44%
  Brightness Variation (-30%): Accuracy = 96.24%
  Brightness Variation (-20%): Accuracy = 97.85%
  Brightness Variation (-10%): Accuracy = 97.85%
  Brightness Variation (  0%): Accuracy = 97.85%
  Brightness Variation (+10%): Accuracy = 98.39%
  Brightn

## 10. Archive Checkpoints and Results for Local Download

In [10]:
import shutil
shutil.make_archive('attcm_alex_results', 'zip', '.', 'results')
shutil.make_archive('attcm_alex_checkpoints', 'zip', '.', 'checkpoints')
print("Checkpoints and Results successfully zipped to attcm_alex_checkpoints.zip and attcm_alex_results.zip!")

Checkpoints and Results successfully zipped to attcm_alex_checkpoints.zip and attcm_alex_results.zip!


In [11]:
import sys

log_file = open("training_results.txt", "w")

class Tee:
    def __init__(self, *files):
        self.files = files

    def write(self, data):
        for f in self.files:
            f.write(data)
            f.flush()

    def flush(self):
        for f in self.files:
            f.flush()

sys.stdout = Tee(sys.__stdout__, log_file)

In [13]:
from google.colab import files

files.download("training_results.txt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>